# MP4: Model Evaluation & Business Impact

**MajsterPlus — Customer Lapse Prediction**

In this mini-project, you will:
1. Construct a cost matrix for the reactivation campaign
2. Calculate profit per record at threshold=0.5
3. Compare against baseline strategies (contact everyone / contact nobody)
4. Optimize the classification threshold for maximum profit
5. Perform lift analysis

**CRISP-DM Phase**: Evaluation

---

## 0. Setup & Reproducibility

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Library version assertions
import sklearn, pandas as pd
assert sklearn.__version__.startswith(("1.4", "1.5")), (
    f"scikit-learn {sklearn.__version__} — expected 1.4.x or 1.5.x"
)
assert pd.__version__.startswith("2."), (
    f"pandas {pd.__version__} — expected 2.x"
)
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} — expected <2.0"
)
print(f"sklearn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}")
print(f"Random seed: {SEED}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Load Checkpoint from MP3

In [ ]:
import hashlib, pickle
from pathlib import Path

# Colab detection
try:
    import google.colab
    DATA_DIR = Path("/content/drive/MyDrive/PUM/2. data")
    CHECKPOINT_DIR = Path("/content/drive/MyDrive/PUM/checkpoints")
except ImportError:
    DATA_DIR = Path("../2. data")
    CHECKPOINT_DIR = Path("../checkpoints")

checkpoint_file = CHECKPOINT_DIR / "mp3_checkpoint.pkl"

if checkpoint_file.exists():
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
    print(f"Loaded checkpoint from MP3: {list(checkpoint.keys())}")
else:
    print(f"No checkpoint found at {checkpoint_file}")
    print("You can still proceed by running MP3 first, or loading raw data.")
    checkpoint = None

In [ ]:
if checkpoint is not None:
    X_train = checkpoint["X_train"]
    X_test = checkpoint["X_test"]
    y_train = checkpoint["y_train"]
    y_test = checkpoint["y_test"]
    feature_names = checkpoint["feature_names"]
    gender_test = checkpoint.get("gender_test")
    lr_model = checkpoint.get("lr_model")
    rf_model = checkpoint.get("rf_model")
    y_prob_lr = checkpoint.get("y_prob_lr")
    y_prob_rf = checkpoint.get("y_prob_rf")
else:
    raise FileNotFoundError(
        "No checkpoint found. Please run MP3 solution first."
    )

print(f"Loaded models and predictions from MP3")
print(f"Test set size: {len(y_test)}")

## 2. Business Context

MajsterPlus is considering a targeted reactivation campaign:
- **Campaign cost per customer**: 80 PLN (voucher + operational costs)
- **Voucher value**: 50 PLN
- **Expected revenue if reactivated**: Median total_spend of lapsed customers in test set

The cost matrix:
| | Actually Lapsed (1) | Actually Active (0) |
|---|---|---|
| **Predicted Lapsed (1)** — contacted | Revenue - Cost (TP) | -Cost (FP) |
| **Predicted Active (0)** — not contacted | 0 (FN) | 0 (TN) |

In [ ]:
CAMPAIGN_COST = 80  # PLN per customer contacted

# Calculate expected revenue from reactivation
# We need total_spend from the original data for lapsed test customers
# Load raw data to get total_spend values
import hashlib
cust_raw = pd.read_csv(DATA_DIR / "customers.csv")
cust_raw["total_spend_numeric"] = (
    cust_raw["total_spend"]
    .str.replace("PLN ", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Get median total_spend of lapsed customers in test set
lapsed_test_idx = y_test[y_test == 1].index
# Map back to original customer rows
lapsed_test_spend = cust_raw.loc[
    cust_raw.index.isin(lapsed_test_idx), "total_spend_numeric"
]
EXPECTED_REVENUE = lapsed_test_spend.median()
print(f"Campaign cost per customer: {CAMPAIGN_COST} PLN")
print(f"Expected revenue if reactivated (median lapsed spend): {EXPECTED_REVENUE:.2f} PLN")

## 3. Cost Matrix Construction

**Task**: Define a function that computes total campaign profit given true labels and predictions.

Think about the four cases: TP, FP, FN, TN — which ones have non-zero financial impact?

- **True Positive** (lapsed, contacted): Revenue - Cost
- **False Positive** (active, contacted): -Cost
- **False Negative** (lapsed, not contacted): 0
- **True Negative** (active, not contacted): 0

In [ ]:
# TODO: Define a function compute_profit(y_true, y_pred, revenue, cost)
# that returns (total_profit, tp_count, fp_count, fn_count, tn_count).
#
# Steps:
# 1. Convert inputs to numpy arrays
# 2. Count TP, FP, FN, TN using boolean masks
# 3. Compute profit = TP * (revenue - cost) + FP * (-cost)
# 4. Return profit and all four counts

# YOUR CODE HERE


### Think About This

Notice the asymmetry in the cost matrix: a True Positive gains you `(Revenue - 80)` PLN, while a False Positive costs you 80 PLN flat. The net gain per TP is only ~60 PLN while the loss per FP is 80 PLN.

**TODO**: What does this imbalance mean for how aggressively the model should predict "lapsed"? Write your reasoning in a new markdown cell below (or as a comment).

In [ ]:
# TODO: Write your reasoning here as a comment or print statement.
# Consider: If the FP penalty (80 PLN) exceeds the TP net gain (~60 PLN),
# should the model be aggressive (predict many positives) or conservative
# (predict fewer, more confident positives)?

# YOUR ANSWER HERE


## 4. Profit at Threshold = 0.5

**Task**: Calculate the profit for both models (LogReg, RF) at the default
threshold of 0.5. Also calculate profit per record.

In [ ]:
# TODO: Convert probability predictions to binary predictions at threshold 0.5
# for both Logistic Regression and Random Forest.
# Then use your compute_profit function to calculate profit for each.
# Print: TP, FP, FN, TN counts, total profit, and profit per record.

# YOUR CODE HERE


## 5. Baseline Comparison

**Task**: Calculate profit for two naive strategies:
1. **Contact everyone**: Predict all customers as lapsed
2. **Contact nobody**: Predict all customers as active

In [ ]:
# TODO: Create two arrays:
# - y_all_ones: all predictions = 1 (contact everyone)
# - y_all_zeros: all predictions = 0 (contact nobody)
# Compute profit for each baseline strategy.

# YOUR CODE HERE


### Think About This

The "contact everyone" strategy loses a significant amount of money.

**TODO**: Why does contacting everyone lose so much money? Think about the class balance — approximately 80% of customers are active, so approximately 80% of your contacts are False Positives at -80 PLN each. Calculate the approximate FP loss explicitly and compare it to the TP gain.

In [ ]:
# TODO: Show the math explicitly:
# - How many contacts are FPs? What is the total FP loss?
# - How many contacts are TPs? What is the total TP gain?
# - Why does the FP loss overwhelm the TP gain?

# YOUR CODE HERE


## 6. Threshold Optimization

**Task**: Loop through thresholds from 0.05 to 0.95 (step 0.05) and compute
profit at each threshold for both models. Plot profit vs. threshold.

In [ ]:
# TODO: Sweep thresholds using np.arange(0.05, 1.0, 0.05).
# For each threshold, convert probabilities to binary predictions
# and compute profit for both LR and RF.
# Store results in lists, then plot both curves on the same chart.
# Add a horizontal line for the "contact everyone" baseline.

# YOUR CODE HERE


## 7. Optimal Threshold Analysis

**Task**: Identify the optimal threshold for each model and analyze the predictions at that threshold.

In [ ]:
# TODO: Find the threshold that maximizes profit for each model.
# Print: optimal threshold, maximum profit, number of customers contacted,
# TP count, FP count, and profit per record at the optimal threshold.

# YOUR CODE HERE


### Think About This

The optimal threshold is likely below 0.5. Why? A lower threshold means predicting MORE customers as lapsed (higher recall, lower precision). At what point does the additional FP cost outweigh the TP gain?

**TODO**: At the optimal threshold, compute:
- (a) The profit from TPs = TP count × (revenue - cost)
- (b) The loss from FPs = FP count × cost
- (c) The net profit = (a) - (b)

Is the margin thin? What would happen if the campaign cost increased by just 10 PLN?

In [ ]:
# TODO: Break down the profit at the optimal threshold into
# TP gain and FP loss components. Discuss the margin.

# YOUR CODE HERE


## 8. Lift Analysis

**Task**: Create a cumulative gains (lift) curve showing how much of the
lapsed customers we capture by contacting the top X% of customers
(ranked by predicted probability).

In [ ]:
# TODO: For the Random Forest model:
# 1. Sort test customers by predicted probability (descending)
# 2. Calculate cumulative number of true lapsed customers captured
# 3. Plot % customers contacted (x-axis) vs % lapsed captured (y-axis)
# 4. Add a diagonal line representing "no model" (random) baseline
# 5. Print the % of lapsed customers captured at 10%, 20%, 30%, and 50% contact levels

# YOUR CODE HERE


### Think About This

**TODO**: At the 20% contact level, you capture a certain percentage of lapsed customers. Is this good enough? What is the business cost of missing the remaining lapsed customers who were not contacted?

Consider the trade-off: contacting more customers increases campaign cost (more FPs) but captures more lapsed customers (more TPs). Where is the sweet spot?

In [ ]:
# TODO: Discuss the trade-off between coverage and cost.
# What fraction of lapsed customers are we willing to miss
# in order to keep the campaign profitable?

# YOUR ANSWER HERE


## 9. Annual Profit Estimate

**Task**: Extrapolate the test-set profit to the full customer base to
estimate annual campaign impact.

In [ ]:
# TODO: The test set is ~20% of the total customer base (5,000 customers
# before outlier removal). Compute test_fraction = len(y_test) / 5000.
# Scale the optimal profit to estimate annual campaign value.
# Also compare: what would the "contact everyone" strategy cost annually?

# YOUR CODE HERE


## 10. Save Checkpoint for MP5

In [ ]:
import pickle

CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_mp4 = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "feature_names": feature_names,
    "gender_test": gender_test,
    "lr_model": lr_model,
    "rf_model": rf_model,
    "y_prob_lr": y_prob_lr,
    "y_prob_rf": y_prob_rf,
    "CAMPAIGN_COST": CAMPAIGN_COST,
    "EXPECTED_REVENUE": EXPECTED_REVENUE,
    # Add your results:
    # "optimal_threshold_rf": optimal_threshold_rf,
    # "optimal_profit_rf": optimal_profit_rf,
}

with open(CHECKPOINT_DIR / "mp4_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint_mp4, f)
print(f"Checkpoint saved: {CHECKPOINT_DIR / 'mp4_checkpoint.pkl'}")

---
*End of MP4 — Continue to MP5: Model Comparison & Final Recommendation*